## 1. Setup

### 1.0 Google Drive Sync

In [46]:
# Adicionar o conteúdo do drive no caminho do sistema

#from google.colab import drive
#drive.mount('/content/drive', force_remount=True)

In [47]:
# modificar para o diretorio que contem a pasta do repositorio ao rodar esse notebook
# fazemos isso para ele conseguir ler a pasta /datasets que precisa estar dentro do diretório que contem esse notebook
%cd /content/drive/MyDrive/projetos-if702/projeto-churn/

!dir

[WinError 3] The system cannot find the path specified: '/content/drive/MyDrive/projetos-if702/projeto-churn/'
c:\Users\User\Desktop\Redes neurais\projetos-if702\projeto-churn
 Volume in drive C has no label.
 Volume Serial Number is 92E5-FB3A

 Directory of c:\Users\User\Desktop\Redes neurais\projetos-if702\projeto-churn

06/05/2026  09:53 PM    <DIR>          .
06/05/2026  09:53 PM    <DIR>          ..
06/05/2026  07:56 PM             4,428 0_preprocessamento.ipynb
06/05/2026  07:58 PM           207,845 1_MLP.ipynb
06/04/2026  04:14 PM           395,387 2_XGBoost.ipynb
06/04/2026  04:14 PM             9,732 churn_preprocessing.py
06/05/2026  07:57 PM    <DIR>          datasets
06/05/2026  10:06 PM           373,528 KAN.ipynb
06/05/2026  10:21 PM            27,246 ml_utils.py
06/05/2026  09:53 PM    <DIR>          model
06/04/2026  04:14 PM    <DIR>          models
06/04/2026  04:14 PM    <DIR>          results
06/04/2026  04:14 PM    <DIR>          searches
06/04/2026  02:42 PM      

In [48]:
import sys

# Colocar o caminho exato da pasta onde está a pasta referente ao repositório do projeto
# O caminho base é sempre '/content/drive/MyDrive/' se criar um atalho no "Meu Drive" para a pasta sincronizada
caminho_projeto = '/content/drive/MyDrive/projetos-if702/projeto-churn'

# Adiciona a pasta no path do Python, caso ainda não esteja lá
if caminho_projeto not in sys.path:
    sys.path.append(caminho_projeto)

In [49]:

# Fazer recarregamento dos modulos locais.
# Assim não precisa reiniciar o kernel do colab toda vez que modificar alguma coisa nos arquivos .py

import importlib
sys.modules['imp'] = importlib

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1.1 Importar Libs

In [50]:
# Importação das bibliotecas necessárias
import pandas as pd
import optuna 
import optuna.visualization as optuna_vis
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import os
import json
from sklearn.metrics import (classification_report, confusion_matrix, 
                           accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
from scipy.stats import uniform, loguniform, randint
from joblib import dump, load
import warnings


warnings.filterwarnings('ignore')

# Importar funções dos módulos customizados
import torch
from sklearn.model_selection import train_test_split
from scikitplot.metrics import plot_ks_statistic
from joblib import dump, load

from ml_utils import (
    train_model, evaluate_model, ks_test,
    build_hyperparameter_space, print_hyperparameter_space,
    run_optuna_search, get_loss_fn, _preprocess_model_params,
    DEFAULT_SCORING_METRIC, AVAILABLE_METRICS_SEARCH, AVAILABLE_LOSSES,
    RANDOM_STATE_MODEL, RANDOM_STATE_SAMPLE,
)
from churn_preprocessing import load_split_datasets, get_X_y_from_split, OUTPUT_PREFIX, TARGET_COL
try:
    from kan import KAN
except:
    print("❌ KANs não encontrado. Instale com: pip install pykan")

# Configurações de plotagem
plt.rcParams['figure.figsize'] = [12, 8]
sns.set_style("whitegrid")

print("Bibliotecas importadas com sucesso!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device.type.upper())

Bibliotecas importadas com sucesso!
Pandas: 2.3.1
NumPy: 2.3.2
Scikit-learn: 1.7.2
Device: CUDA


### 1.2 Configuração do Modelo

In [51]:
# Configuração do modelo e hiperparâmetros
MODEL_NAME = "KAN"
MODEL_CLASS = KAN

module_name = getattr(MODEL_CLASS, "__module__", "")
print(f"Modelo configurado: {MODEL_NAME}")
print(module_name)

print("Hiperparâmetros disponíveis na classe do modelo:")
kan_params = [
    "width",
    "grid",
    "k",
    "noise_scale",
    "noise_scale_base",
    "base_fun",
    "symbolic_enabled",
    "bias_trainable",
    "sp_trainable",
    "sb_trainable",
    "lr",
    "batch_size",
    "steps",
    "lamb",
    "lamb_l1",
    "lamb_entropy",
    "lamb_coef",
    "lamb_coefdiff",
    "seed",
]
for param in kan_params:
    print(f"  - {param}")

Modelo configurado: KAN
kan.MultKAN
Hiperparâmetros disponíveis na classe do modelo:
  - width
  - grid
  - k
  - noise_scale
  - noise_scale_base
  - base_fun
  - symbolic_enabled
  - bias_trainable
  - sp_trainable
  - sb_trainable
  - lr
  - batch_size
  - steps
  - lamb
  - lamb_l1
  - lamb_entropy
  - lamb_coef
  - lamb_coefdiff
  - seed


## 2. Carregamento e Preparação dos Dados

In [52]:
# Carregamento e preparação inicial dos dados
print("=== CARREGAMENTO DOS DATASETS ===")

train_data, val_data, test_data = load_split_datasets(prefix=OUTPUT_PREFIX)
X_train, y_train, X_val, y_val, X_test, y_test = get_X_y_from_split(train_data, val_data, test_data, TARGET_COL)

X_train_scaled = X_train
X_test_scaled = X_test

print(f"Dataset de treino: {train_data.shape}")
print(f"Dataset de validação: {val_data.shape}")
print(f"Dataset de teste: {test_data.shape}")
print(f"Features: {X_train_scaled.shape[1]}")

print("\nDistribuição das classes:")
print("Treino:", pd.Series(y_train).value_counts().to_dict())
print("Val:", pd.Series(y_val).value_counts().to_dict())
print("Teste:", pd.Series(y_test).value_counts().to_dict())

=== CARREGAMENTO DOS DATASETS ===
Dataset de treino: (5174, 20)
Dataset de validação: (2586, 20)
Dataset de teste: (1762, 20)
Features: 19

Distribuição das classes:
Treino: {0: 2587, 1: 2587}
Val: {0: 1293, 1: 1293}
Teste: {0: 1294, 1: 468}


In [53]:
train_data.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,PaperlessBilling_Yes,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaymentMethod,Churn
0,0.0,-0.181818,0.660499,0.051516,0.0,0.0,1.0,1.0,1.0,2.0,1.0,0.0,2.0,2.0,0.0,2.0,2.0,1.0,1.0,0
1,0.0,0.886364,-0.552846,0.591772,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0,2.0,2.0,0.0,0.0,2.0,1.0,2.0,0
2,0.0,0.681818,-0.318475,0.662118,1.0,1.0,1.0,1.0,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,0.0,2.0,1.0,0
3,0.0,-0.363636,0.239978,-0.249371,1.0,1.0,0.0,1.0,1.0,2.0,1.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,2.0,1
4,1.0,-0.068182,0.086347,0.092913,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,1


## 3. Sampling para Busca de Hiperparâmetros

### 3.0 Sampling para busca

In [54]:

SAMPLE_SIZE = 0.05  # % of training data for hyperparameter search
# Amostra estratificada do dataset de treino
_, X_sample, _, y_sample = train_test_split(
    X_train_scaled, y_train,
    test_size=SAMPLE_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE_SAMPLE
)

print(f"Dataset original de treino: {X_train_scaled.shape[0]:,} amostras")
print(f"Amostra para busca de hiperparâmetros: {X_sample.shape[0]:,} amostras")
print(f"Redução: {(1 - X_sample.shape[0]/X_train_scaled.shape[0])*100:.1f}%")

print("\nDistribuição das classes na amostra:")
print("Amostra:", pd.Series(y_sample).value_counts().to_dict())
print("Original:", pd.Series(y_train).value_counts().to_dict())


Dataset original de treino: 5,174 amostras
Amostra para busca de hiperparâmetros: 259 amostras
Redução: 95.0%

Distribuição das classes na amostra:
Amostra: {0: 130, 1: 129}
Original: {0: 2587, 1: 2587}


### 3.1 Configs da Busca Optuna

In [55]:
print("Métrica padrão de avaliação na busca de hiperparâmetros caso não especificada outra:", DEFAULT_SCORING_METRIC)
print("Métricas disponíveis para a busca de hiperparâmetros:\n", AVAILABLE_METRICS_SEARCH)
print("Funções de perda disponíveis para modelos PyTorch:\n", AVAILABLE_LOSSES)

Métrica padrão de avaliação na busca de hiperparâmetros caso não especificada outra: ks
Métricas disponíveis para a busca de hiperparâmetros:
 ['accuracy', 'precision', 'recall', 'f1', 'val_loss', 'auroc', 'ks']
Funções de perda disponíveis para modelos PyTorch:
 ['cross_entropy', 'bce_with_logits', 'mse']


In [56]:
# Optuna
n_trials = 100
scoring_metric = 'ks'
loss_fn = get_loss_fn('bce_with_logits')

# modelos PyTorch somente
early_stopping_patience = 10
early_stopping_delta = 0.001
training_epochs_per_trial = 10

### 3.2 Definição do espaço de hiperparâmetros

In [57]:
param_space = build_hyperparameter_space(
    MODEL_CLASS,
    overrides={
        "hidden_dim": {"type": "int", "values": [1, 4]},
        "grid": {"type": "int", "values": [3, 20]},
        "k": {"type": "categorical", "values": [2, 3, 4, 5]},
        "lr": {"type": "float", "values": [1e-4, 1e-1, None, True]},
    },
)

print_hyperparameter_space(param_space)

Hyperparameter space:
- hidden_dim: type=int, args=[1, 4], kwargs={}
- grid: type=int, args=[3, 20], kwargs={}
- k: type=categorical, values=[2, 3, 4, 5]
- lr: type=float, args=[0.0001, 0.1], kwargs={'log': True}


In [58]:
import torch
from kan import KAN

n_features = 19
test_model = KAN(
    width=[n_features, 2, 1],
    grid=5,
    k=3,
    grid_range=[-2.0, 2.0],
    seed=42,
)

x = torch.randn(8, n_features)
out = test_model(x)
print("Output shape:", out.shape)
print("Output:", out)

checkpoint directory created: ./model
saving model version 0.0
Output shape: torch.Size([8, 1])
Output: tensor([[ 0.1295],
        [-0.1888],
        [ 0.0629],
        [-0.0582],
        [ 0.1053],
        [ 0.2061],
        [ 0.1911],
        [ 0.0678]], grad_fn=<AddBackward0>)


In [59]:
print(type(X_sample))
print(X_sample.shape)
print(X_sample.ndim)

<class 'numpy.ndarray'>
(259, 19)
2


In [63]:
import subprocess
result = subprocess.run(['pip', 'show', 'pykan'], capture_output=True, text=True)
print(result.stdout)

Name: pykan
Version: 0.2.8
Summary: Kolmogorov Arnold Networks
Home-page: 
Author: Ziming Liu
Author-email: zmliu@mit.edu
License: 
Location: C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: 
Required-by: 



In [64]:
from torch.utils.data import DataLoader, TensorDataset
import torch

X_t = torch.tensor(X_sample.astype('float32'))
y_t = torch.tensor(y_sample.astype('float32'))
ds = TensorDataset(X_t, y_t)
dl = DataLoader(ds, batch_size=32, shuffle=True)

test_model = KAN(
    width=[19, 2, 1],
    grid=5,
    k=3,
    grid_range=[-2.0, 2.0],
    seed=42,
)

batch_X, batch_y = next(iter(dl))
print("batch_X shape antes do forward:", batch_X.shape)
out = test_model(batch_X)
print("Output shape:", out.shape)

checkpoint directory created: ./model
saving model version 0.0
batch_X shape antes do forward: torch.Size([32, 19])
Output shape: torch.Size([32, 1])


In [65]:
import inspect
from kan import KAN
print(inspect.getsource(KAN.train))

    def train(self: T, mode: bool = True) -> T:
        r"""Set the module in training mode.

        This has an effect only on certain modules. See the documentation of
        particular modules for details of their behaviors in training/evaluation
        mode, i.e., whether they are affected, e.g. :class:`Dropout`, :class:`BatchNorm`,
        etc.

        Args:
            mode (bool): whether to set training mode (``True``) or evaluation
                         mode (``False``). Default: ``True``.

        Returns:
            Module: self
        """
        if not isinstance(mode, bool):
            raise ValueError("training mode is expected to be boolean")
        self.training = mode
        for module in self.children():
            module.train(mode)
        return self



### 3.3 Executar Busca de Hiperparâmetros

In [60]:
print(f"=== BUSCA DE HIPERPARÂMETROS - {MODEL_NAME} ===")
print(f"Iniciando busca de hiperparâmetros para {MODEL_NAME}...")
print(f"Trials: {n_trials}")
print(f"Usando amostra de {X_sample.shape[0]:,} exemplos\n")
print(f"Hiperparâmetros no espaço: {list(param_space.keys())}")

direction = "minimize" if scoring_metric in {"val_loss", "loss"} else "maximize"
study = run_optuna_search(
    model_class=MODEL_CLASS,
    space=param_space,
    X_train=X_sample,
    y_train=y_sample,
    X_val=X_val,
    y_val=y_val,
    n_trials=n_trials,
    direction=direction,
    scoring_metric=scoring_metric,
    device=device,
    epochs=training_epochs_per_trial,
    patience=early_stopping_patience,
    min_delta=early_stopping_delta,
    loss_fn=loss_fn,
)

print(f"\n--- RESULTADOS {MODEL_NAME} ---")
print("Melhores hiperparâmetros encontrados:")
for param, value in study.best_params.items():
    print(f"  {param}: {value}")
print(f"\nMelhor score ({scoring_metric}): {study.best_value:.6f}")

[I 2026-06-05 22:22:21,205] A new study created in memory with name: no-name-38c625fd-014d-4bc8-ae4e-71b0ca51cae8
[W 2026-06-05 22:22:21,224] Trial 0 failed with parameters: {'hidden_dim': 4, 'grid': 15, 'k': 5, 'lr': 0.011839026096637787} because of the following error: IndexError('too many indices for tensor of dimension 1').
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "c:\Users\User\Desktop\Redes neurais\projetos-if702\projeto-churn\ml_utils.py", line 734, in _objective
        **objective_kwargs: Any,
                   ^^^^^^^^^^^^^
    ...<8 lines>...
                y_train,
    
  File "c:\Users\User\Desktop\Redes neurais\projetos-if702\projeto-churn\ml_utils.py", line 686, in optuna_objective
    val_loss = None
    
    ...<12 lines>...
            batch_size=batch_size,
    
  File "c:\Users\User\Desktop\Redes neu

=== BUSCA DE HIPERPARÂMETROS - KAN ===
Iniciando busca de hiperparâmetros para KAN...
Trials: 100
Usando amostra de 259 exemplos

Hiperparâmetros no espaço: ['hidden_dim', 'grid', 'k', 'lr']
checkpoint directory created: ./model
saving model version 0.0
X_np shape: (259, 19)
y_np shape: (259,)
X_np shape: (2586, 19)
y_np shape: (2586,)
batch_X shape: torch.Size([19])
batch_X ndim: 1


IndexError: too many indices for tensor of dimension 1

### 3.4 Visualizar histórico da busca

In [ ]:
optuna_vis.plot_param_importances(study)

In [ ]:
optuna_vis.plot_optimization_history(study, target_name=scoring_metric)

In [ ]:
optuna_vis.plot_parallel_coordinate(study, params=list(param_space.keys()), target_name=scoring_metric)

In [ ]:
print(f"=== MELHORES CONFIGURAÇÕES ENCONTRADAS - {MODEL_NAME} ===")

trials_df = study.trials_dataframe()
if "state" in trials_df.columns:
    trials_df = trials_df[trials_df["state"] == "COMPLETE"]
sorted_df = trials_df.sort_values("value", ascending=(direction == "minimize"))

print(f"\nTop configurações (de {len(sorted_df)} trials completos):")
print(sorted_df.head(10).to_string(index=False))

if not sorted_df.empty:
    print(f"\nEstatísticas dos scores encontrados:")
    print(f"  Média: {sorted_df['value'].mean():.4f}")
    print(f"  Mediana: {sorted_df['value'].median():.4f}")
    print(f"  Desvio padrão: {sorted_df['value'].std():.4f}")
    print(f"  Min: {sorted_df['value'].min():.4f}")
    print(f"  Max: {sorted_df['value'].max():.4f}")

## 4. Salvar Resultados de Busca

In [ ]:
SEARCH_RESULTS_FOLDER = "searches"
os.makedirs(SEARCH_RESULTS_FOLDER, exist_ok=True)

trials_path = os.path.join(SEARCH_RESULTS_FOLDER, f"{MODEL_NAME.lower()}_optuna_trials.csv")
best_params_path = os.path.join(SEARCH_RESULTS_FOLDER, f"{MODEL_NAME.lower()}_optuna_best.json")

trials_df.to_csv(trials_path, index=False)
with open(best_params_path, "w") as f:
    json.dump(study.best_params, f, indent=2, default=str)

print("Resultados da busca salvos:")
print(trials_path)
print(best_params_path)

In [ ]:
loaded_best_params = None
loaded_trials_df = None

## 5. Carregar Resultado de Busca (Opcional)

In [ ]:
if os.path.exists(trials_path):
    loaded_trials_df = pd.read_csv(trials_path)
if os.path.exists(best_params_path):
    with open(best_params_path, "r") as f:
        loaded_best_params = json.load(f)

In [ ]:
# Plotar a história da busca a partir dos resultados carregados
if loaded_trials_df is not None and "value" in loaded_trials_df.columns:
    plot_df = loaded_trials_df.sort_values("number") if "number" in loaded_trials_df.columns else loaded_trials_df
    plt.figure(figsize=(8, 4))
    plt.plot(plot_df["value"].values, marker="o")
    plt.title(f"Histórico da busca ({scoring_metric})")
    plt.xlabel("Trial")
    plt.ylabel("Score")
    plt.tight_layout()
    plt.show()

## 6. Definir Melhores Params e Score de Busca

In [ ]:
# Definir Melhores Parâmetros para Uso Posterior
if loaded_best_params is not None:
    best_params = loaded_best_params
    if loaded_trials_df is not None and "value" in loaded_trials_df.columns:
        best_score = (
            loaded_trials_df["value"].min()
            if direction == "minimize"
            else loaded_trials_df["value"].max()
        )
    else:
        best_score = study.best_value
    print(f"✅ Usando parâmetros carregados: {best_params}")
    print(f"✅ Melhor score carregado: {best_score:.6f}")
else:
    best_params = study.best_params
    best_score = study.best_value
    print(f"✅ Usando parâmetros da busca atual: {best_params}")
    print(f"✅ Melhor score da busca atual: {best_score:.6f}")

## 7. Treinar Modelo Final e Salvar

In [ ]:
best_params_model = _preprocess_model_params(KAN, best_params.copy(), X_train_scaled)
lr = best_params.get("lr", 1e-3)

best_model = KAN(**best_params_model, seed=RANDOM_STATE_MODEL, device=device)

optimizer = torch.optim.Adam(best_model.parameters(), lr=lr)

train_result = train_model(...)
best_model = train_result["model"]

In [ ]:
MODELS_FOLDER = "models"
os.makedirs(MODELS_FOLDER, exist_ok=True)

model_path = os.path.join(MODELS_FOLDER, f'{MODEL_NAME.lower().replace(" ", "_")}_model.pt')
torch.save(best_model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

## 8. Avaliação Final e Salvamento dos Resultados

In [ ]:
# Carregar modelo (Opcional)
n_features = X_train_scaled.shape[1]
hidden_dim = best_params["hidden_dim"]

loaded_model = KAN(
    width=[n_features, hidden_dim, 1],
    grid=best_params.get("grid", 5),
    k=best_params.get("k", 3),
    grid_range=best_params.get("grid_range", [-2.0, 2.0]),
    seed=RANDOM_STATE_MODEL,
    device=device,
)
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.to(device)

In [ ]:
print(f"=== AVALIAÇÃO E SALVAMENTO DOS RESULTADOS - {MODEL_NAME} ===")

# Criar pastas se não existirem
RESULTS_FOLDER = "results"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

# Avaliação completa do modelo
print("\nAvaliando:", MODEL_NAME)

# Usar datasets completos para avaliação final
X_train_eval = X_train_scaled
y_train_eval = y_train
X_test_eval = X_test_scaled
y_test_eval = y_test

# Avaliar modelo usando função do módulo
train_metrics, y_train_pred, y_train_scores = evaluate_model(
    best_model, X_train_eval, y_train_eval, device=device
 )
test_metrics, y_test_pred, y_test_scores = evaluate_model(
    best_model, X_test_eval, y_test_eval, device=device
 )

### 8.1 Classification Report

In [ ]:
print(classification_report(y_test_eval, y_test_pred, zero_division=0))

In [ ]:
result = ks_test(y_test_eval, y_test_scores)
probas = result["probas"]
print(f"KS Statistic: {result['ks_stat']:.4f}")
ax = plot_ks_statistic(y_test_eval, probas)

### 8.2 Visualize Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_test_eval, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title(f'{MODEL_NAME} - Confusion Matrix (Test Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### 8.3 Save Final Results

In [ ]:
# Salvar resultados finais
RESULTS_FOLDER = "results"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

final_results = {
    "model_name": MODEL_NAME,
    "best_params": best_params,
    "best_score": best_score,
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
}

final_path = os.path.join(RESULTS_FOLDER, f"{MODEL_NAME.lower()}_final_results.json")
with open(final_path, "w") as f:
    json.dump(final_results, f, indent=2, default=str)

# Mostrar resumo final
print(f"\n--- RESUMO FINAL {MODEL_NAME} ---")
print(f"Score da busca: {best_score:.6f}")
print(f"KS Statistic Teste: {test_metrics['ks_stat']:.4f}")
print(f"AUC-ROC Teste: {test_metrics['auroc']:.4f}")
print(f"F1-Score Teste: {test_metrics['f1']:.4f}")
print(f"Acurácia Teste: {test_metrics['accuracy']:.4f}")
print(f"Precisão Teste: {test_metrics['precision']:.4f}")
print(f"Recall Teste: {test_metrics['recall']:.4f}")


print(f"Resultados salvos em: {RESULTS_FOLDER}/")
print(f"Arquivo final: {final_path}")